In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os

print(os.listdir("/kaggle/input"))


In [ ]:
import os

path = "/kaggle/input/network-intrusion-dataset"

files = [os.path.join(path, f) for f in os.listdir(path)]

print("Total files:", len(files))


In [ ]:
import pandas as pd

df_list = []

for file in files:
    print("Reading:", file)
    df = pd.read_csv(file)
    df_list.append(df)

full_df = pd.concat(df_list, ignore_index=True)

print("Final Shape:", full_df.shape)


In [ ]:
full_df

In [ ]:
import numpy as np

print("Before cleaning:", full_df.shape)

# Remove duplicates
full_df.drop_duplicates(inplace=True)

# Replace infinity values with NaN
full_df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Drop missing values
full_df.dropna(inplace=True)

print("After cleaning:", full_df.shape)


In [ ]:
print(full_df.columns)


In [ ]:
full_df.columns = full_df.columns.str.strip()


In [ ]:
print(full_df['Label'].value_counts())


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
full_df['Label'] = le.fit_transform(full_df['Label'])

print(full_df['Label'].value_counts())


In [ ]:
X = full_df.drop('Label', axis=1)
y = full_df['Label']

print("Features shape:", X.shape)
print("Target shape:", y.shape)


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
print(full_df['Label'].value_counts())


In [ ]:
import matplotlib.pyplot as plt

full_df['Label'].value_counts().plot(kind='bar')
plt.title("Class Distribution")
plt.show()


In [ ]:
from sklearn.model_selection import train_test_split

X_train_real, X_test_real, y_train_real, y_test_real = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, balanced_accuracy_score

rf_real = RandomForestClassifier(
    n_estimators=100,
    n_jobs=-1,
    random_state=42
)

rf_real.fit(X_train_real, y_train_real)

y_pred_real = rf_real.predict(X_test_real)

print("Accuracy:", accuracy_score(y_test_real, y_pred_real))
print("Balanced Accuracy:", balanced_accuracy_score(y_test_real, y_pred_real))
print(classification_report(y_test_real, y_pred_real))


In [ ]:
import pandas as pd

X_train_df = pd.DataFrame(X_train_real, columns=X.columns)

train_df = X_train_df.copy()
train_df['Label'] = y_train_real.values

print("Training DataFrame shape:", train_df.shape)


In [ ]:
!pip install ctgan


In [ ]:
from ctgan import CTGAN

ctgan = CTGAN(epochs=50)
ctgan.fit(train_df)

print("CTGAN training complete.")


In [ ]:
import torch
print(torch.cuda.is_available())


In [ ]:
train_df_small = train_df.sample(n=100000, random_state=42)
ctgan.fit(train_df_small)
